## 1. 왜 ONNX Runtime에서 TensorRT로 가는가

`05_onnx_export.ipynb`에서 ONNX Runtime CUDA EP로 **102ms → 5.2× 가속**을 달성했습니다.  
TensorRT는 그 다음 단계입니다.

| 항목 | ONNX Runtime CUDA EP | TensorRT |
|---|---|---|
| 최적화 방식 | 범용 CUDA 커널 | **모델 구조 분석 후 GPU 전용 커널 재컴파일** |
| 레이어 퓨전 | 제한적 | Conv+BN+ReLU 등 자동 퓨전 |
| FP16 지원 | 있음 (제한적) | 있음 (엔진 빌드 시 명시) |
| 이식성 | 높음 (다양한 하드웨어) | 낮음 (빌드한 GPU에서만 동작) |
| 속도 | 빠름 | **더 빠름** |

핵심 차이: TensorRT는 **배포 대상 GPU에 맞춰 모델을 재컴파일**하므로,  
ONNX Runtime보다 같은 하드웨어에서 추가 가속이 가능합니다.

## 2. GTX 1660 Super 주의사항

TU116 아키텍처(Turing) 기반으로 다음을 확인하고 가야 합니다:

| 기능 | GTX 1660 Super | 비고 |
|---|---|---|
| FP32 TensorRT | ✔ | 기본 지원 |
| FP16 TensorRT | ✔ | CUDA 코어 기반 (Tensor Core 없음) |
| INT8 TensorRT | ⚠ 제한적 | Tensor Core 부재 → 속도 이득 미미, 정확도 손실 위험 있어 이 노트북에서는 제외 |

<br/>

> **FP16이 왜 빠른가 (Tensor Core 없어도):**  
> FP16은 FP32 대비 데이터 크기가 절반 → 메모리 대역폭 소모 감소 → 캐시 효율 향상.  
> GTX 1660 Super에서도 FP16이 FP32보다 빠르지만, RTX 시리즈(Tensor Core 보유)만큼 극적이진 않습니다.

## 3. 환경 설정 (Windows)

### 3-1. 기존 환경 확인

In [2]:
# ── 환경 확인 (venv 활성화 불필요: 이 커널이 곧 그 환경) ──
import sys, subprocess, torch

# 1) 현재 파이썬이 어느 환경인지 → venv 맞는지 경로로 확인
print(f"Python 실행 경로 : {sys.executable}")
#   ...\venv_battery\Scripts\python.exe 처럼 나오면 venv 안에서 도는 것

# 2) GPU / 드라이버 확인 (nvidia-smi)
print("\n── nvidia-smi ──")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

# 3) PyTorch가 보는 CUDA
print("── PyTorch CUDA ──")
print(f"torch        : {torch.__version__}")
print(f"CUDA 빌드     : {torch.version.cuda}")          # 12.x 기대
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")    # True 기대
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")  # GTX 1660 Super 기대

Python 실행 경로 : d:\02.study\part4_wj\Battery\Battery_Project\venv_battery\Scripts\python.exe

── nvidia-smi ──
Wed Jun 10 10:36:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.57                 Driver Version: 581.57         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1660 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| 41%   35C    P8             17W /  125W |     943MiB /   6144MiB |     10%      Default |
|                             

CPU 빌드로 되어있길래 확인 해보겠습니다.

In [5]:
# 1. torch 패밀리 버전 일괄 확인
import subprocess
print(subprocess.run(["pip", "list"], capture_output=True, text=True).stdout
    .__class__ and "\n".join(
        l for l in subprocess.run(["pip","list"],capture_output=True,text=True).stdout.splitlines()
        if any(k in l.lower() for k in ["torch","tensorrt","onnx","ultralytics","nvidia","cuda"])))

nvidia-cublas-cu12        12.9.2.10
nvidia-cuda-nvrtc-cu12    12.9.86
nvidia-cuda-runtime-cu12  12.9.79
nvidia-cudnn-cu12         9.22.0.52
onnx                      1.21.0
onnx-ir                   0.2.1
onnxruntime-gpu           1.26.0
onnxscript                0.7.0
torch                     2.11.0
torchvision               0.26.0
ultralytics               8.4.62
ultralytics-thop          2.0.20


In [7]:
# 2. 전체 환경 리포트 (가장 정보 많음)
import torch.utils.collect_env as e; e.main()

PyTorch version: 2.11.0+cpu
Is debug build: False
CUDA used to build PyTorch: Could not collect
ROCM used to build PyTorch: N/A

OS: Microsoft Windows 11 Home (10.0.26200 64비트)
GCC version: (MinGW-W64 x86_64-ucrt-posix-seh, built by Brecht Sanders, r8) 13.2.0
Clang version: Could not collect
CMake version: version 3.29.2
Libc version: N/A

Python version: 3.11.14 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 18:30:03) [MSC v.1929 64 bit (AMD64)] (64-bit runtime)
Python platform: Windows-10-10.0.26200-SP0
Is CUDA available: False
CUDA runtime version: Could not collect
CUDA_MODULE_LOADING set to: N/A
GPU models and configuration: GPU 0: NVIDIA GeForce GTX 1660 SUPER
Nvidia driver version: 581.57
cuDNN version: Could not collect
Is XPU available: False
HIP runtime version: N/A
MIOpen runtime version: N/A
Is XNNPACK available: True
Caching allocator config: N/A

CPU:
Name: 13th Gen Intel(R) Core(TM) i5-13400F
Manufacturer: GenuineIntel
Family: 205
Architecture: 9
ProcessorType: 3
Dev

In [8]:
# 3. torch가 언제 설치됐는지 (시점 = 원인 추적의 핵심)
import torch, os, glob, datetime
sp = os.path.dirname(os.path.dirname(torch.__file__))
for d in glob.glob(os.path.join(sp, "torch*-*.dist-info")):
    t = datetime.datetime.fromtimestamp(os.path.getmtime(d))
    print(t.strftime("%Y-%m-%d %H:%M"), os.path.basename(d))

2026-05-12 01:24 torch-2.11.0.dist-info
2026-05-12 01:49 torchvision-0.26.0.dist-info


In [9]:
# 4. 예전 CUDA 휠이 캐시에 남아있나 (복구 난이도 판단)
import subprocess
print(subprocess.run(["pip","cache","list","torch"],capture_output=True,text=True).stdout)

No locally built wheels cached.



원인: 5월 12일 torch 업그레이드가 CPU 로 교체가 되었습니다.

| 단서 | 의미 |
| :--- | :--- |
| `torch==2.11.0+cpu`, `torchvision 0.26.0` | venv의 현재 torch가 CPU 빌드 |
| dist-info 날짜 2026-05-12 01:24 / 01:49 | 그날 torch·torchvision이 (재)설치됨 |
| `[conda] pytorch 2.5.1 py3.11_cuda12.1_cudnn9_0` | 예전에 CUDA로 잘 돌던 그 torch (conda base에 아직 남아있음) |
| `ultralytics 8.4.62`, `onnxruntime-gpu 1.26.0` 존재 | nb10(YOLOv8) 작업하며 패키지 설치한 흔적 |
| `nvidia-cublas/cudnn/cuda-runtime-cu12` (pip) | 이건 onnxruntime-gpu가 끌어온 것 (torch 게 아님) |


TensorRT를 위주로 진행하고 싶기에 같은 버전(2.11.0)으로 CUDA 빌드만 갈아끼워 다른 노트북 영향 최소화하겠습니다.

```powershell
pip install --force-reinstall torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu128
```

그리고 커널 재시작으로 오류를 수정하였습니다.

In [1]:
import torch
print(torch.__version__)             
print(torch.version.cuda)            
print(torch.cuda.is_available())      
print(torch.cuda.get_device_name(0))

2.11.0+cu128
12.8
True
NVIDIA GeForce GTX 1660 SUPER


GPU torch 가 잡혔으니 **TensorRT** 를 설치하겠습니다.

```powershell
d:\02.study\part4_wj\Battery\Battery_Project\venv_battery\Scripts\python.exe -m pip install tensorrt cuda-python
```

In [1]:
import tensorrt as trt
print(f"TensorRT: {trt.__version__}")

TensorRT: 11.0.0.114


## TensorRT 11 호환성

이 노트북은 **TensorRT 10.x 이전** 기준으로 작성됐는데, 이 PC에는 **TensorRT 11.0.0.114**(최신)가 설치돼 있습니다. 11.0에서 API가 많이 바뀌어, 아래처럼 코드를 수정하며 진행했습니다.

| 옛 코드 (문서) | TRT 11에서 생긴 문제 | 대응 |
|---|---|---|
| `parser.parse(f.read())` | 외부 가중치 `.onnx.data`를 못 읽음 | **`parser.parse_from_file()`** 로 교체 |
| `builder.platform_has_fast_fp16` | 속성 자체가 제거됨 | 체크 삭제 |
| `BuilderFlag.FP16` | **enum에서 제거됨** (FP16/INT8 플래그 없음, TF32만 존재) | **FP16 포기 → FP32로 진행** |
| `create_network(1<<EXPLICIT_BATCH)` | explicit batch가 기본값이 됨 | `create_network(0)` |
| `execute_async_v2(bindings)` | 제거됨 | **`execute_async_v3` + `set_tensor_address`** (추론 셀) |


### 앞으로의 진행
아래에서 **FP32 TensorRT 엔진**을 빌드하고, 다음 3가지 추론 속도를 비교합니다:<br/>
`ONNX CPU` → `ONNX CUDA EP` → `TensorRT FP32`. 마지막 표의 가속 배수를 확인하겠습니다.

# 4. 엔진 빌드 함수

In [6]:
from pathlib import Path

PROJECT   = Path(r'D:\02.study\part4_wj\Battery\Battery_Project')
MODEL_DIR = PROJECT / 'models'

# 입력 ONNX (외부 .onnx.data 동반 — 같은 폴더에 있어야 함)
DEEPLAB_ONNX = MODEL_DIR / 'battery_deeplab_v1.onnx'

# 출력 TensorRT 엔진
DEEPLAB_FP32 = MODEL_DIR / 'deeplab_fp32.engine'
DEEPLAB_FP16 = MODEL_DIR / 'deeplab_fp16.engine'

assert DEEPLAB_ONNX.exists(), f"파일 없음: {DEEPLAB_ONNX}"
assert (MODEL_DIR / 'battery_deeplab_v1.onnx.data').exists(), "외부 가중치(.onnx.data) 없음"
print("경로 확인 완료:", DEEPLAB_ONNX.name)

경로 확인 완료: battery_deeplab_v1.onnx


In [2]:
import tensorrt as trt
from pathlib import Path

TRT_LOGGER = trt.Logger(trt.Logger.WARNING)


def build_engine(onnx_path: Path, engine_path: Path, fp16: bool = False) -> None:
    """ONNX(외부 .onnx.data 포함) → TensorRT 11 엔진 빌드 후 저장."""
    builder = trt.Builder(TRT_LOGGER)
    network = builder.create_network(0)
    parser  = trt.OnnxParser(network, TRT_LOGGER)
    config  = builder.create_builder_config()

    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 2 << 30)  # 2GB

    if fp16:
        config.set_flag(trt.BuilderFlag.FP16)
        print("FP16 모드 활성화")

    # 외부 가중치(.onnx.data) 때문에 parse_from_file 사용 (parse(f.read()))
    if not parser.parse_from_file(str(onnx_path)):
        for i in range(parser.num_errors):
            print(f"파싱 오류: {parser.get_error(i)}")
        raise RuntimeError(f"ONNX 파싱 실패: {onnx_path.name}")

    print(f"빌드 시작: {onnx_path.name} → {'FP16' if fp16 else 'FP32'} (최초 수 분 소요)")
    serialized = builder.build_serialized_network(network, config)
    if serialized is None:
        raise RuntimeError("엔진 빌드 실패 — VRAM 부족 시 workspace 줄이기")

    engine_path.write_bytes(serialized)
    print(f"저장 완료: {engine_path.name} ({engine_path.stat().st_size/1e6:.1f} MB)")

# 5. DeepLabV3+ 엔진 빌드

In [7]:
import time

t0 = time.time()
build_engine(DEEPLAB_ONNX, DEEPLAB_FP32, fp16=False)
print(f"FP32 빌드 시간: {time.time()-t0:.1f}초")

빌드 시작: battery_deeplab_v1.onnx → FP32 (최초 수 분 소요)
저장 완료: deeplab_fp32.engine (176.2 MB)
FP32 빌드 시간: 72.1초


In [8]:
# IO 텐서 정보 확인 (추론 코드 맞추는 데 필요)
rt = trt.Runtime(TRT_LOGGER)
eng = rt.deserialize_cuda_engine(DEEPLAB_FP32.read_bytes())
for i in range(eng.num_io_tensors):
    n = eng.get_tensor_name(i)
    print(f"{eng.get_tensor_mode(n).name:6} | {n:20} | "
        f"shape={tuple(eng.get_tensor_shape(n))} | dtype={eng.get_tensor_dtype(n)}")

INPUT  | input                | shape=(1, 3, 513, 513) | dtype=DataType.FLOAT
OUTPUT | logits               | shape=(1, 3, 513, 513) | dtype=DataType.FLOAT


## 6. 추론 & latency 벤치마크

이제 **같은 입력**에 대해 세 가지 추론 속도를 잽니다:

1. **ONNX Runtime (CPU)** — 기준선 (가장 느림)
2. **ONNX Runtime (CUDA EP)** — GPU 범용 가속 (05번 노트북에서 102ms 기록)
3. **TensorRT FP32** — GPU 전용 커널로 재컴파일한 엔진 (가장 빠를 것으로 기대)

TRT 11에서는 추론도 `execute_async_v3` + `set_tensor_address` 방식으로 바뀌었고, GPU 메모리 관리는 (pycuda 대신) **cuda-python** 으로 합니다. 아래 셀의 `TRTRunner` 클래스가 그 처리를 담당하고, 마지막에 **가속 배수 표**가 출력됩니다.

> **읽는 포인트:** latency는 이미지 "내용"과 무관하므로(같은 크기면 동일), 더미 입력으로 측정해도 수치는 동일합니다. 워밍업 10회 후 50회 평균을 잽니다.

In [14]:
import numpy as np, time
import tensorrt as trt
import onnxruntime as ort
try:
    from cuda import cudart                          # cuda-python 12.x 이하
except ImportError:
    from cuda.bindings import runtime as cudart      # cuda-python 12.6+/13.x


def _chk(ret):
    if ret[0] != cudart.cudaError_t.cudaSuccess:
        raise RuntimeError(f"CUDA 오류: {cudart.cudaGetErrorString(ret[0])}")
    return ret[1:]


class TRTRunner:
    """TensorRT 11 엔진 추론기 (cuda-python, 기본 스트림 사용)."""
    def __init__(self, engine_path):
        with open(engine_path, "rb") as f:
            self.engine = trt.Runtime(trt.Logger(trt.Logger.WARNING)).deserialize_cuda_engine(f.read())
        self.ctx = self.engine.create_execution_context()
        self.io = []
        for i in range(self.engine.num_io_tensors):
            name  = self.engine.get_tensor_name(i)
            is_in = self.engine.get_tensor_mode(name) == trt.TensorIOMode.INPUT
            shape = tuple(self.engine.get_tensor_shape(name))
            dtype = trt.nptype(self.engine.get_tensor_dtype(name))
            nbytes = int(np.prod(shape)) * np.dtype(dtype).itemsize
            dptr  = _chk(cudart.cudaMalloc(nbytes))[0]
            self.ctx.set_tensor_address(name, dptr)             # TRT 11: 주소 등록
            self.io.append(dict(name=name, is_in=is_in, shape=shape,
                                dtype=dtype, nbytes=nbytes, dptr=dptr))
        self.inp = next(t for t in self.io if t["is_in"])
        self.out = next(t for t in self.io if not t["is_in"])
        self.host_out = np.empty(self.out["shape"], dtype=self.out["dtype"])

    def infer(self, x):
        x = np.ascontiguousarray(x, dtype=self.inp["dtype"])
        _chk(cudart.cudaMemcpyAsync(self.inp["dptr"], x.ctypes.data, self.inp["nbytes"],
            cudart.cudaMemcpyKind.cudaMemcpyHostToDevice, 0))
        self.ctx.execute_async_v3(0)                            # TRT 11: v2 → v3
        _chk(cudart.cudaMemcpyAsync(self.host_out.ctypes.data, self.out["dptr"], self.out["nbytes"],
            cudart.cudaMemcpyKind.cudaMemcpyDeviceToHost, 0))
        _chk(cudart.cudaDeviceSynchronize())
        return self.host_out


def bench(fn, x, warmup=10, runs=50):
    for _ in range(warmup): fn(x)
    t0 = time.perf_counter()
    for _ in range(runs): fn(x)
    return (time.perf_counter() - t0) / runs * 1000

In [ ]:
# 08_onnx_export.ipynb 참고
import os
from pathlib import Path

NVIDIA_BIN = Path(r'D:\02.study\part4_wj\Battery\Battery_Project\venv_battery\Lib\site-packages\nvidia')
for sub in ['cudnn', 'cuda_runtime', 'cublas']:
    d = str(NVIDIA_BIN / sub / 'bin')
    if Path(d).is_dir():
        os.add_dll_directory(d)                      # Python 3.8+ 공식 방법
        os.environ['PATH'] = d + os.pathsep + os.environ['PATH']
        print(f'CUDA DLL dir added: {d}')
    else:
        print(f'(없음) {d}')

CUDA DLL dir added: D:\02.study\part4_wj\Battery\Battery_Project\venv_battery\Lib\site-packages\nvidia\cudnn\bin
CUDA DLL dir added: D:\02.study\part4_wj\Battery\Battery_Project\venv_battery\Lib\site-packages\nvidia\cuda_runtime\bin
CUDA DLL dir added: D:\02.study\part4_wj\Battery\Battery_Project\venv_battery\Lib\site-packages\nvidia\cublas\bin


In [16]:
# 입력: 엔진 입력 shape에 맞춘 더미 (latency는 내용 무관)
trt_run = TRTRunner(str(DEEPLAB_FP32))
inp = np.random.rand(*trt_run.inp["shape"]).astype(np.float32)

results = {}
sess_cpu = ort.InferenceSession(str(DEEPLAB_ONNX), providers=["CPUExecutionProvider"])
results["ONNX CPU FP32"] = bench(lambda x: sess_cpu.run(None, {sess_cpu.get_inputs()[0].name: x}), inp)

sess_gpu = ort.InferenceSession(str(DEEPLAB_ONNX),
                                providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
results["ONNX CUDA EP FP32"] = bench(lambda x: sess_gpu.run(None, {sess_gpu.get_inputs()[0].name: x}), inp)

results["TensorRT FP32"] = bench(trt_run.infer, inp)

base = results["ONNX CPU FP32"]
print(f"\n{'방식':<22}{'latency':>12}{'가속':>10}")
print("-" * 44)
for k, v in results.items():
    print(f"{k:<22}{v:>9.1f} ms{base / v:>8.1f}x")


방식                         latency        가속
--------------------------------------------
ONNX CPU FP32             548.9 ms     1.0x
ONNX CUDA EP FP32         102.1 ms     5.4x
TensorRT FP32              82.9 ms     6.6x


# 7. TensorRT 정확도 검증

In [18]:
import numpy as np
from PIL import Image

# 전처리 (torchvision 없이 numpy로 — torch DLL 충돌 회피)
mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

img_path = next((PROJECT / 'battery_image').glob('RGB_cell_cylindrical_*.png'))
img = Image.open(img_path).convert('RGB').resize((513, 513), Image.BILINEAR)
arr = np.asarray(img, dtype=np.float32) / 255.0      # HWC, 0~1
arr = (arr - mean) / std                             # 정규화
x = arr.transpose(2,0,1)[None].astype(np.float32)    # (1, 3, 513, 513)

# 기준: ONNX(CPU)
sess = ort.InferenceSession(str(DEEPLAB_ONNX), providers=['CPUExecutionProvider'])
onnx_out = sess.run(None, {sess.get_inputs()[0].name: x})[0]

# 비교: TensorRT FP32
trt_out = trt_run.infer(x)

diff = np.abs(onnx_out - trt_out)
agree = (onnx_out.argmax(1) == trt_out.argmax(1)).mean()
print(f"max abs diff : {diff.max():.3e}")
print(f"argmax 일치   : {agree * 100:.3f}%   (목표 ≥ 99.9%)")

max abs diff : 1.097e-04
argmax 일치   : 100.000%   (목표 ≥ 99.9%)


## 8. 결론 — TensorRT 변환 결과 요약

### 8-1. 추론 속도 (GTX 1660 Super · 입력 513×513 · 워밍업 10 / 측정 50회 평균)

| 추론 방식 | latency | CPU 대비 | 비고 |
|---|---|---|---|
| ONNX Runtime CPU FP32 | 548.9 ms | 1.0× | 베이스라인 |
| ONNX Runtime CUDA EP FP32 | 102.1 ms | 5.4× | GPU 범용 가속 (08번 102ms 재현) |
| **TensorRT FP32** | **82.9 ms** | **6.6×** | GPU 전용 커널로 재컴파일 |

> ONNX CUDA EP 대비 TensorRT가 약 19ms(≈1.23×) 추가 단축. 레이어 퓨전·커널 자동 튜닝 효과로 해석.

### 8-2. 변환 정확도 (무손실 확인)

| 항목 | 결과 | 기준 |
|---|---|---|
| ONNX ↔ TensorRT max abs diff | 1.10e-04 | < 1e-3 |
| argmax(세그멘테이션 마스크) 일치율 | 100.000% | ≥ 99.9% |

→ FP32 TensorRT 엔진은 ONNX와 **픽셀 단위로 동일한 출력**을 생성. 가속에 따른 정확도 손실 **0**.

### 8-3. 회고 / 배운 점
- 설치된 TensorRT가 **11.0**(가이드 기준 10.x)이라, `parse_from_file` / `create_network(0)` / `execute_async_v3` / `set_tensor_address` 등 신규 API로 코드를 이식.
- TRT 11은 FP16 빌더 플래그를 제거(strongly-typed 전환)하여 본 노트북은 **FP32**로 진행. FP16은 ONNX를 FP16으로 변환하는 별도 단계가 필요.
- ONNX Runtime CUDA EP가 DLL 미탐색으로 CPU 폴백하던 문제를 `os.add_dll_directory`로 해결(08번과 동일 패턴). `get_providers()`로 폴백을 진단.

### 8-4. 범위 명시
- 본 노트북의 검증 범위는 **"ONNX → TensorRT 변환 · 가속 · 엔진 출력 등가성"** 입니다.
- 결함 검출 **모델 품질**(과검/미검, GT 대비 성능)은 `09_re_verification.ipynb`의 범위입니다.